# Block 2: Feature Engineering

This notebook converts the cleaned label dataset from Block 1 into model-ready tables.

Inputs:

- `outputs/cleaned/cleaned_events_with_labels.csv`
- `outputs/cleaned/leakage_policy.csv`

Outputs:

- `outputs/feature_engineering/model_ready_road_closure.csv`
- `outputs/feature_engineering/model_ready_duration_band.csv`
- `outputs/feature_engineering/feature_columns.json`
- `outputs/feature_engineering/feature_engineering_summary.csv`

Design goals:

- Use only prediction-time-safe fields.
- Create strong temporal, spatial, categorical, text, and prior-history features.
- Use prior-only historical closure statistics so the current row's label never leaks into its own features.

## 1. Imports and Project Paths

In [1]:
from pathlib import Path
import json
import re
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 140)
pd.set_option('display.max_rows', 100)

cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [cwd, *cwd.parents] if (p / 'outputs' / 'cleaned').exists()), cwd)
CLEANED_PATH = PROJECT_ROOT / 'outputs' / 'cleaned' / 'cleaned_events_with_labels.csv'
POLICY_PATH = PROJECT_ROOT / 'outputs' / 'cleaned' / 'leakage_policy.csv'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'feature_engineering'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Project root:', PROJECT_ROOT)
print('Cleaned data:', CLEANED_PATH)
print('Leakage policy:', POLICY_PATH)
print('Output dir:', OUTPUT_DIR)

Project root: D:\Python\Gridlock\Phase 2\theme 2
Cleaned data: D:\Python\Gridlock\Phase 2\theme 2\outputs\cleaned\cleaned_events_with_labels.csv
Leakage policy: D:\Python\Gridlock\Phase 2\theme 2\outputs\cleaned\leakage_policy.csv
Output dir: D:\Python\Gridlock\Phase 2\theme 2\outputs\feature_engineering


## 2. Load Cleaned Data and Leakage Policy

In [2]:
df = pd.read_csv(CLEANED_PATH, low_memory=False)
policy = pd.read_csv(POLICY_PATH)

safe_raw_cols = policy.loc[policy['use_as_model_feature'].eq(True), 'column'].tolist()

print('Cleaned shape:', df.shape)
print('Safe raw columns from policy:', len(safe_raw_cols))
display(df.head())
display(policy.groupby(['role', 'use_as_model_feature']).size().reset_index(name='columns'))

Cleaned shape: (8173, 72)
Safe raw columns from policy: 33


,id,event_type,latitude,longitude,endlatitude,endlongitude,address,end_address,event_cause,requires_road_closure,start_datetime,end_datetime,status,authenticated,modified_datetime,map_file,direction,description,veh_type,veh_no,corridor,priority,cargo_material,reason_breakdown,age_of_truck,created_date,route_path,client_id,created_by_id,last_modified_by_id,assigned_to_police_id,citizen_accident_id,comment,police_station,meta_data,kgid,resolved_at_address,resolved_at_latitude,resolved_at_longitude,closed_by_id,closed_datetime,resolved_by_id,resolved_datetime,gba_identifier,zone,junction,start_datetime_ist,end_datetime_ist,created_date_ist,modified_datetime_ist,closed_datetime_ist,resolved_datetime_ist,valid_start_coordinate,has_start_location,has_raw_end_location,has_resolved_location,target_road_closure,duration_minutes_from_end,duration_minutes_from_resolved,duration_minutes_from_closed,duration_minutes,duration_source,valid_duration_label,duration_band,report_lag_minutes,report_lag_is_negative,start_hour,start_dayofweek,start_month,is_weekend,is_peak_hour,is_night
0,FKID000000,unplanned,13.040004,77.518099,NaN,NaN,"Mumbai Bengaluru Highway, Jalahalli Cross Junc...",NaN,vehicle_breakdown,False,2024-03-07 17:01:48.111000+00:00,NaN,closed,yes,2024-03-07 19:35:47.871698+00:00,NaN,NaN,s m circle in coming man track,lcv,FKN00GL0000,Tumkur Road,High,NaN,NaN,NaN,2024-03-07 17:03:51.164032+00:00,NaN,1,FKUSR00000,FKUSR00001,NaN,NaN,NaN,Peenya,NaN,FKKG000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-03-07 22:31:48.111000+05:30,NaN,2024-03-07 22:33:51.164032+05:30,2024-03-08 01:05:47.871698+05:30,NaN,NaN,1,1,0,0,0,NaN,NaN,NaN,NaN,NaN,0,NaN,2.050884,0,22.0,3.0,2024-03,0,0,1
1,FKID000001,unplanned,12.921876,77.645158,NaN,NaN,"19th Main Road, Heavie Halcyon, Agara, HSR Lay...",NaN,vehicle_breakdown,False,2024-01-30 04:07:24.173000+00:00,NaN,resolved,yes,2024-01-30 04:17:46.828979+00:00,NaN,NaN,Starting problem,heavy_vehicle,FKN00GL0001,ORR East 1,High,NaN,NaN,NaN,2024-01-30 04:08:22.954979+00:00,NaN,1,FKUSR00002,FKUSR00001,NaN,NaN,NaN,HSR Layout,NaN,FKKG000001,"19th Main Road, Heavie Halcyon, Agara, HSR Lay...",12.921876,77.645158,NaN,NaN,FKUSR00002,2024-01-30 04:17:46.828355+00:00,NaN,NaN,NaN,2024-01-30 09:37:24.173000+05:30,NaN,2024-01-30 09:38:22.954979+05:30,2024-01-30 09:47:46.828979+05:30,NaN,2024-01-30 09:47:46.828355+05:30,1,1,0,1,0,NaN,10.377589,NaN,10.377589,resolved,1,short,0.979700,0,9.0,1.0,2024-01,0,1,0
2,FKID000002,unplanned,12.955622,77.585708,NaN,NaN,"Lalbagh Main Road, Dr Sri Shantaveera Swami Ci...",NaN,others,False,2023-11-11 06:18:03.343000+00:00,NaN,closed,yes,2024-01-30 04:56:03.282003+00:00,NaN,NaN,ಊರ್ವಶಿ ಜಂಕ್ಷನ್ ನಲ್ಲಿ ಒಳಚರಂಡಿ ಚೇಂಬರ್ ಗೆ ಹೊಸದಾಗಿ...,NaN,NaN,Non-corridor,Low,NaN,NaN,NaN,2023-11-11 06:20:00.989398+00:00,NaN,1,FKUSR00003,FKUSR00001,NaN,NaN,NaN,Wilson Garden,NaN,FKKG000002,NaN,NaN,NaN,FKUSR00003,2024-01-30 04:56:03.281509+00:00,NaN,NaN,Bengaluru Central Corporation,Central Zone 2,UrvashiJunction,2023-11-11 11:48:03.343000+05:30,NaN,2023-11-11 11:50:00.989398+05:30,2024-01-30 10:26:03.282003+05:30,2024-01-30 10:26:03.281509+05:30,NaN,1,1,0,0,0,NaN,NaN,115117.998975,115117.998975,closed,0,NaN,1.960773,0,11.0,5.0,2023-11,1,0,0
3,FKID000003,unplanned,13.006147,77.579435,13.006239,77.579516,"Sankey Road, Bashyam Circle, Sadashiva Nagar, ...","Sankey Road, Palace Orchard Upper, Sadashiva N...",tree_fall,True,2024-03-07 17:56:55.061000+00:00,NaN,closed,yes,2024-03-14 07:42:05.550050+00:00,NaN,NaN,tree fall,NaN,NaN,Non-corridor,Low,NaN,NaN,NaN,2024-03-07 17:58:56.696892+00:00,NaN,1,FKUSR00004,FKUSR00001,NaN,NaN,NaN,Sadashivanagar,NaN,FKKG000003,NaN,NaN,NaN,FKUSR00004,2024-03-14 07:42:05.549440+00:00,NaN,NaN,NaN,NaN,NaN,2024-03-07 23:26:55.061000+05:30,NaN,2024-03-07 23:28:56.696892+05:30,2024-03-14 13:12:05.550050+05:30,2024-03-14 13:12:05.549440+05:30,NaN,1,1,1,0,1,NaN,NaN,9465.174807,9465.174807,closed,0,NaN,2.027265,0,23.0,3.0,2024-03,0,0,1
4,FKID000004,unplanned,12.953980,77.585233,NaN,NaN,"Lalbagh Fort Roa

,role,use_as_model_feature,columns
0,future_or_label_only,False,26
1,id_or_admin,False,11
2,prediction_time_candidate,True,33
3,target_or_raw_target,False,2


## 3. Helper Functions

In [3]:
def clean_text_series(series):
    return (
        series.fillna('unknown')
        .astype(str)
        .str.strip()
        .str.lower()
        .replace({'': 'unknown', 'nan': 'unknown', 'none': 'unknown', 'null': 'unknown'})
    )


def group_rare(series, min_count=25, other_label='other_rare'):
    s = clean_text_series(series)
    counts = s.value_counts(dropna=False)
    common = set(counts[counts >= min_count].index)
    return s.where(s.isin(common), other_label)


def haversine_km(lat1, lon1, lat2, lon2):
    lat1 = np.radians(lat1.astype(float))
    lon1 = np.radians(lon1.astype(float))
    lat2 = np.radians(lat2.astype(float))
    lon2 = np.radians(lon2.astype(float))
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 6371.0 * 2 * np.arcsin(np.sqrt(a))


def prior_group_stats(frame, group_col, target_col='target_road_closure'):
    # Uses only rows before the current row in chronological order.
    grouped = frame.groupby(group_col, dropna=False)[target_col]
    prior_count = grouped.cumcount()
    prior_positive = grouped.cumsum() - frame[target_col].fillna(0)
    global_prior = frame[target_col].expanding().mean().shift(1).fillna(frame[target_col].mean())
    prior_rate = (prior_positive / prior_count.replace(0, np.nan)).fillna(global_prior)
    return prior_count.astype(float), prior_rate.astype(float)


def sanitize_columns(columns):
    clean_cols = []
    seen = {}
    for col in columns:
        clean = re.sub(r'[^0-9a-zA-Z_]+', '_', str(col).strip().lower())
        clean = re.sub(r'_+', '_', clean).strip('_')
        if not clean:
            clean = 'feature'
        if clean in seen:
            seen[clean] += 1
            clean = f'{clean}_{seen[clean]}'
        else:
            seen[clean] = 0
        clean_cols.append(clean)
    return clean_cols

## 4. Base Feature Frame

This block creates numeric, temporal, spatial, text, and business-rule features. It avoids future columns by construction.

In [4]:
feat = pd.DataFrame(index=df.index)

# Numeric coordinates and location quality.
for col in ['latitude', 'longitude', 'valid_start_coordinate', 'has_start_location']:
    if col in df.columns:
        feat[col] = pd.to_numeric(df[col], errors='coerce')

# Bengaluru city center approximation: Vidhana Soudha / CBD area.
if {'latitude', 'longitude'}.issubset(df.columns):
    feat['distance_to_city_center_km'] = haversine_km(
        pd.to_numeric(df['latitude'], errors='coerce').fillna(12.9716),
        pd.to_numeric(df['longitude'], errors='coerce').fillna(77.5946),
        pd.Series(12.9716, index=df.index),
        pd.Series(77.5946, index=df.index),
    )
    feat['location_grid'] = (
        pd.to_numeric(df['latitude'], errors='coerce').round(3).astype(str)
        + '_'
        + pd.to_numeric(df['longitude'], errors='coerce').round(3).astype(str)
    ).replace({'nan_nan': 'unknown'})

# Datetime-derived safe features.
start_dt = pd.to_datetime(df.get('start_datetime_ist', df.get('start_datetime')), errors='coerce')
created_dt = pd.to_datetime(df.get('created_date_ist', df.get('created_date')), errors='coerce')

feat['start_hour'] = start_dt.dt.hour
feat['start_dayofweek'] = start_dt.dt.dayofweek
feat['start_weekofyear'] = start_dt.dt.isocalendar().week.astype(float)
feat['is_weekend'] = feat['start_dayofweek'].isin([5, 6]).astype(int)
feat['is_peak_hour'] = feat['start_hour'].isin([7, 8, 9, 17, 18, 19, 20]).astype(int)
feat['is_night'] = feat['start_hour'].isin([22, 23, 0, 1, 2, 3, 4, 5]).astype(int)
feat['hour_sin'] = np.sin(2 * np.pi * feat['start_hour'].fillna(0) / 24)
feat['hour_cos'] = np.cos(2 * np.pi * feat['start_hour'].fillna(0) / 24)
feat['day_sin'] = np.sin(2 * np.pi * feat['start_dayofweek'].fillna(0) / 7)
feat['day_cos'] = np.cos(2 * np.pi * feat['start_dayofweek'].fillna(0) / 7)

report_lag = (created_dt - start_dt).dt.total_seconds() / 60
feat['report_lag_minutes_clipped'] = report_lag.clip(lower=-60, upper=1440)
feat['report_lag_hours_clipped'] = (report_lag / 60).clip(lower=-1, upper=24)
feat['report_lag_is_negative'] = (report_lag < 0).fillna(False).astype(int)
feat['reporting_delay_minutes'] = report_lag.clip(lower=0, upper=1440)

# Text features from description/address/comment-like fields available at creation.
description = df.get('description', pd.Series('', index=df.index)).fillna('').astype(str)
address = df.get('address', pd.Series('', index=df.index)).fillna('').astype(str)
combined_text = (description + ' ' + address).str.lower()

feat['text_length'] = combined_text.str.len()
feat['description_char_length'] = description.str.len()
feat['description_word_count'] = description.str.split().str.len().fillna(0)
feat['has_non_ascii_text'] = combined_text.str.contains(r'[^\x00-]', regex=True).astype(int)
feat['has_kannada_text'] = combined_text.str.contains(r'[ಀ-೿]', regex=True).astype(int)

keyword_patterns = {
    'has_accident_word': r'accident|crash|hit|collision|injur',
    'has_breakdown_word': r'breakdown|starting problem|puncture|tyre|engine|truck|lorry',
    'has_water_word': r'water|rain|flood|logging|logged',
    'has_construction_word': r'construction|cement|work|repair|metro|road work',
    'has_event_word': r'rally|festival|procession|event|match|gathering|vip',
    'has_blocked_word': r'block|blocked|closure|closed|obstruction|barricade',
    'has_jam_word': r'jam|slow|congestion|traffic',
    'has_vip_word': r'vip|minister|political|rally',
    'has_location_hint_word': r'junction|circle|cross|road|gate|flyover',
    'has_emergency_word': r'emergency|ambulance|fire|police',
}
for feature_name, pattern in keyword_patterns.items():
    feat[feature_name] = combined_text.str.contains(pattern, regex=True).astype(int)

print('Feature frame shape after base features:', feat.shape)
display(feat.head())

Feature frame shape after base features: (8173, 35)


,latitude,longitude,valid_start_coordinate,has_start_location,distance_to_city_center_km,location_grid,start_hour,start_dayofweek,start_weekofyear,is_weekend,is_peak_hour,is_night,hour_sin,hour_cos,day_sin,day_cos,report_lag_minutes_clipped,report_lag_hours_clipped,report_lag_is_negative,reporting_delay_minutes,text_length,description_char_length,description_word_count,has_non_ascii_text,has_kannada_text,has_accident_word,has_breakdown_word,has_water_word,has_construction_word,has_event_word,has_blocked_word,has_jam_word,has_vip_word,has_location_hint_word,has_emergency_word
0,13.040004,77.518099,1,1,11.249443,13.04_77.518,22.0,3.0,10.0,0,0,1,-0.500000,0.866025,0.433884,-0.900969,2.050884,0.034181,0,2.050884,132,31,7,0,0,0,0,0,0,0,0,0,0,1,0
1,12.921876,77.645158,1,1,7.783945,12.922_77.645,9.0,1.0,5.0,0,1,0,0.707107,-0.707107,0.781831,0.623490,0.979700,0.016328,0,0.979700,108,16,2,0,0,0,1,0,0,0,0,0,0,1,0
2,12.955622,77.585708,1,1,2.021119,12.956_77.586,11.0,5.0,45.0,1,0,0,0.258819,-0.965926,-0.974928,-0.222521,1.960773,0.032680,0,1.960773,219,117,15,1,1,0,0,0,0,0,0,0,0,1,0
3,13.006147,77.579435,1,1,4.178109,13.006_77.579,23.0,3.0,10.0,0,0,1,-0.258819,0.965926,0.433884,-0.900969,2.027265,0.033788,0,2.027265,96,9,2,0,0,0,0,0,0,0,0,0,0,1,0
4,12.953980,77.585233,1,1,2.206553,12.954_77.585,10.0,1.0,5.0,0,0,0,0.500000,-0.866025,0.781831,0.623490,2.393161,0.039886,0,2.393161,151,48,7,1,1,0,0,0,0,0,0,0,0,1,0


## 5. Domain Flags and Grouped Categorical Features

In [5]:
# Raw categoricals available at event creation time.
cat_source_cols = [
    'event_type', 'event_cause', 'direction', 'veh_type', 'corridor', 'priority',
    'cargo_material', 'reason_breakdown', 'police_station', 'zone', 'junction', 'authenticated'
]

for col in cat_source_cols:
    if col not in df.columns:
        df[col] = np.nan

for col in cat_source_cols:
    min_count = 20
    if col in {'junction', 'police_station'}:
        min_count = 30
    if col in {'cargo_material', 'reason_breakdown', 'direction'}:
        min_count = 10
    feat[f'{col}_grouped'] = group_rare(df[col], min_count=min_count)

cause = feat['event_cause_grouped']
veh_type = feat['veh_type_grouped']
event_type = feat['event_type_grouped']

feat['is_planned_event'] = event_type.eq('planned').astype(int)
feat['is_public_or_vip_event'] = cause.str.contains('public_event|procession|vip', regex=True).astype(int)
feat['is_breakdown_event'] = cause.str.contains('breakdown').astype(int)
feat['is_accident_event'] = cause.str.contains('accident').astype(int)
feat['is_weather_or_visibility_event'] = cause.str.contains('water|rain|visibility').astype(int)
feat['is_road_condition_event'] = cause.str.contains('pot_holes|road_conditions|construction').astype(int)

feat['has_vehicle_type'] = ~veh_type.eq('unknown')
feat['has_vehicle_type'] = feat['has_vehicle_type'].astype(int)
feat['is_truck'] = veh_type.str.contains('truck|lorry').astype(int)
feat['is_bus'] = veh_type.str.contains('bus|bmtc|ksrtc').astype(int)
feat['is_two_wheeler'] = veh_type.str.contains('two|bike|scooter').astype(int)
feat['is_heavy_vehicle'] = veh_type.str.contains('heavy|truck|lcv|bus|lorry').astype(int)
feat['has_cargo_material'] = ~feat['cargo_material_grouped'].eq('unknown')
feat['has_cargo_material'] = feat['has_cargo_material'].astype(int)

age = pd.to_numeric(df.get('age_of_truck'), errors='coerce')
feat['age_of_truck'] = age.fillna(age.median() if age.notna().any() else 0).clip(lower=0, upper=50)
feat['truck_age_missing'] = age.isna().astype(int)
feat['truck_age_band'] = pd.cut(
    feat['age_of_truck'],
    bins=[-0.1, 3, 8, 15, 100],
    labels=['new', 'mid', 'old', 'very_old']
).astype(str).replace({'nan': 'unknown'})

# Keep location-grid useful for history/hotspot signal without exploding one-hot columns.
if 'location_grid' in feat.columns:
    feat['location_grid'] = group_rare(feat['location_grid'], min_count=15)

print('Categorical/domain feature shape:', feat.shape)
display(feat[[c for c in feat.columns if c.endswith('_grouped')][:10]].head())

Categorical/domain feature shape: (8173, 62)


,event_type_grouped,event_cause_grouped,direction_grouped,veh_type_grouped,corridor_grouped,priority_grouped,cargo_material_grouped,reason_breakdown_grouped,police_station_grouped,zone_grouped
0,unplanned,vehicle_breakdown,unknown,lcv,tumkur road,high,unknown,unknown,peenya,unknown
1,unplanned,vehicle_breakdown,unknown,heavy_vehicle,orr east 1,high,unknown,unknown,hsr layout,unknown
2,unplanned,others,unknown,unknown,non-corridor,low,unknown,unknown,wilson garden,central zone 2
3,unplanned,tree_fall,unknown,unknown,non-corridor,low,unknown,unknown,sadashivanagar,unknown
4,unplanned,vehicle_breakdown,unknown,private_bus,non-corridor,low,unknown,unknown,wilson garden,unknown


## 6. Interaction Features

These capture traffic-relevant combinations such as cause by corridor and cause by peak period.

In [6]:
def peak_period(hour):
    if pd.isna(hour):
        return 'unknown'
    hour = int(hour)
    if hour in [7, 8, 9]:
        return 'morning_peak'
    if hour in [17, 18, 19, 20]:
        return 'evening_peak'
    if hour in [22, 23, 0, 1, 2, 3, 4, 5]:
        return 'night'
    return 'off_peak'

feat['peak_period'] = feat['start_hour'].apply(peak_period)
feat['event_cause_corridor'] = feat['event_cause_grouped'] + '_' + feat['corridor_grouped']
feat['cause_peak_interaction'] = feat['event_cause_grouped'] + '_' + feat['peak_period']
feat['zone_cause_interaction'] = feat['zone_grouped'] + '_' + feat['event_cause_grouped']
feat['corridor_cause_interaction'] = feat['corridor_grouped'] + '_' + feat['event_cause_grouped']
feat['cause_heavy_vehicle_interaction'] = feat['event_cause_grouped'] + '_heavy_' + feat['is_heavy_vehicle'].astype(str)
feat['corridor_peak_interaction'] = feat['corridor_grouped'] + '_' + feat['peak_period']
feat['start_day_name'] = start_dt.dt.day_name().str.lower().fillna('unknown')
feat['start_month'] = start_dt.dt.to_period('M').astype(str).replace({'NaT': 'unknown'})

interaction_cols = [
    'event_cause_corridor', 'cause_peak_interaction', 'zone_cause_interaction',
    'corridor_cause_interaction', 'cause_heavy_vehicle_interaction', 'corridor_peak_interaction',
]
for col in interaction_cols:
    feat[col] = group_rare(feat[col], min_count=20)

print('Feature shape after interactions:', feat.shape)
display(feat[interaction_cols + ['peak_period', 'start_day_name', 'start_month']].head())

Feature shape after interactions: (8173, 71)


,event_cause_corridor,cause_peak_interaction,zone_cause_interaction,corridor_cause_interaction,cause_heavy_vehicle_interaction,corridor_peak_interaction,peak_period,start_day_name,start_month
0,vehicle_breakdown_tumkur road,vehicle_breakdown_night,unknown_vehicle_breakdown,tumkur road_vehicle_breakdown,vehicle_breakdown_heavy_1,tumkur road_night,night,thursday,2024-03
1,vehicle_breakdown_orr east 1,vehicle_breakdown_morning_peak,unknown_vehicle_breakdown,orr east 1_vehicle_breakdown,vehicle_breakdown_heavy_1,orr east 1_morning_peak,morning_peak,tuesday,2024-01
2,others_non-corridor,others_off_peak,central zone 2_others,non-corridor_others,others_heavy_0,non-corridor_off_peak,off_peak,saturday,2023-11
3,tree_fall_non-corridor,tree_fall_night,unknown_tree_fall,non-corridor_tree_fall,tree_fall_heavy_0,non-corridor_night,night,thursday,2024-03
4,vehicle_breakdown_non-corridor,vehicle_breakdown_off_peak,unknown_vehicle_breakdown,non-corridor_vehicle_breakdown,vehicle_breakdown_heavy_1,non-corridor_off_peak,off_peak,tuesday,2024-01


## 7. Prior-Only Historical Closure Features

Historical risk features are computed in chronological order and use only events that happened before the current row.

In [7]:
chron_order = pd.to_datetime(df.get('created_date', df.get('start_datetime')), errors='coerce')
work = pd.DataFrame({
    'original_index': df.index,
    'chron_order': chron_order,
    'target_road_closure': pd.to_numeric(df['target_road_closure'], errors='coerce').fillna(0),
})

history_group_cols = [
    'event_cause_grouped', 'corridor_grouped', 'zone_grouped', 'junction_grouped',
    'police_station_grouped', 'location_grid', 'event_cause_corridor'
]
for col in history_group_cols:
    work[col] = feat[col].astype(str).fillna('unknown')

work = work.sort_values(['chron_order', 'original_index']).reset_index(drop=True)

for col in history_group_cols:
    prior_count, prior_rate = prior_group_stats(work, col, target_col='target_road_closure')
    work[f'past_count_{col}'] = prior_count
    work[f'past_closure_rate_{col}'] = prior_rate

history_cols = [c for c in work.columns if c.startswith('past_')]
history_features = work[['original_index'] + history_cols].set_index('original_index').sort_index()
feat = feat.join(history_features)

# Rename verbose grouped suffixes to cleaner names.
rename_history = {
    'past_count_event_cause_grouped': 'past_count_event_cause',
    'past_count_corridor_grouped': 'past_count_corridor',
    'past_count_zone_grouped': 'past_count_zone',
    'past_count_junction_grouped': 'past_count_junction',
    'past_count_police_station_grouped': 'past_count_police_station',
    'past_count_location_grid': 'past_count_location_grid',
    'past_count_event_cause_corridor': 'past_count_event_cause_corridor',
    'past_closure_rate_event_cause_grouped': 'past_closure_rate_event_cause',
    'past_closure_rate_corridor_grouped': 'past_closure_rate_corridor',
    'past_closure_rate_zone_grouped': 'past_closure_rate_zone',
    'past_closure_rate_junction_grouped': 'past_closure_rate_junction',
    'past_closure_rate_police_station_grouped': 'past_closure_rate_police_station',
    'past_closure_rate_location_grid': 'past_closure_rate_location_grid',
    'past_closure_rate_event_cause_corridor': 'past_closure_rate_event_cause_corridor',
}
feat = feat.rename(columns=rename_history)

print('Feature shape after historical features:', feat.shape)
display(feat[[c for c in feat.columns if c.startswith('past_')]].head())

Feature shape after historical features: (8173, 85)


,past_count_event_cause,past_closure_rate_event_cause,past_count_corridor,past_closure_rate_corridor,past_count_zone,past_closure_rate_zone,past_count_junction,past_closure_rate_junction,past_count_police_station,past_closure_rate_police_station,past_count_location_grid,past_closure_rate_location_grid,past_count_event_cause_corridor,past_closure_rate_event_cause_corridor
0,3750.0,0.046400,343.0,0.029155,2591.0,0.083365,3525.0,0.076879,139.0,0.035971,14.0,0.071429,288.0,0.017361
1,2594.0,0.048188,113.0,0.106195,690.0,0.068116,1628.0,0.062654,56.0,0.125000,3748.0,0.077375,71.0,0.056338
2,8.0,0.000000,28.0,0.107143,15.0,0.066667,47.0,0.063830,3.0,0.000000,62.0,0.064516,3.0,0.000000
3,101.0,0.425743,2286.0,0.111549,2592.0,0.083333,3526.0,0.076858,209.0,0.066986,5462.0,0.081289,70.0,0.500000
4,2596.0,0.048151,1604.0,0.101621,692.0,0.067919,1965.0,0.086514,51.0,0.176471,3749.0,0.077354,888.0,0.074324


## 8. One-Hot Encode Categoricals

All remaining categorical feature columns are one-hot encoded. Numeric columns are median-imputed.

In [8]:
label_cols = ['target_road_closure', 'duration_band', 'valid_duration_label']

# ── NEW: Cause-level severity prior (static, non-leaking global rates from EDA) ──
CAUSE_CLOSURE_RATE = {
    'vip_movement': 0.80, 'public_event': 0.46, 'protest': 0.40,
    'tree_fall': 0.39, 'construction': 0.26, 'procession': 0.26,
    'road_conditions': 0.12, 'others': 0.09, 'water_logging': 0.09,
    'congestion': 0.04, 'vehicle_breakdown': 0.04, 'accident': 0.03,
    'pot_holes': 0.02, 'other_rare': 0.05, 'unknown': 0.05,
}
feat['cause_base_closure_rate'] = feat['event_cause_grouped'].map(CAUSE_CLOSURE_RATE).fillna(0.05)
feat['cause_severity_rank'] = feat['cause_base_closure_rate'].rank(method='dense', ascending=False)

# ── NEW: Corridor importance flag ──
major_corridors = {
    'mysore road', 'bellary road 1', 'bellary road 2', 'tumkur road',
    'hosur road', 'orr east 1', 'orr east 2', 'orr north 1', 'orr north 2',
    'orr west 1', 'old madras road',
}
feat['corridor_is_major'] = feat['corridor_grouped'].isin(major_corridors).astype(int)

# ── NEW: Priority as numeric ──
feat['priority_is_high'] = (feat['priority_grouped'] == 'high').astype(int)

# ── NEW: Description urgency score (count of multiple keyword hits) ──
keyword_flag_cols = [c for c in feat.columns if c.startswith('has_') and c.endswith('_word')]
feat['description_urgency_score'] = feat[keyword_flag_cols].sum(axis=1)

# ── NEW: Authenticated as numeric ──
feat['is_authenticated'] = (feat['authenticated_grouped'] == 'yes').astype(int)

# ── IMPROVED: Target-encode categoricals instead of one-hot ──
# Use prior-only closure rate as the encoding (already computed for some).
# For categoricals without prior stats, compute smoothed global mean encoding.

categorical_feature_cols = feat.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()
numeric_feature_cols = [c for c in feat.columns if c not in categorical_feature_cols]

# Target-encode all remaining grouped categoricals using smoothed prior closure rate
global_mean = df['target_road_closure'].mean()
SMOOTHING_FACTOR = 30

target_encoded_cols = []
for col in categorical_feature_cols:
    col_vals = feat[col].astype(str).fillna('unknown')
    # Compute group-level stats from target
    group_stats = df.groupby(col_vals)['target_road_closure'].agg(['mean', 'count'])
    smoothed = (group_stats['count'] * group_stats['mean'] + SMOOTHING_FACTOR * global_mean) / (group_stats['count'] + SMOOTHING_FACTOR)
    te_col_name = f'{col}_te'
    feat[te_col_name] = col_vals.map(smoothed).fillna(global_mean)
    target_encoded_cols.append(te_col_name)

numeric_features = feat[numeric_feature_cols + target_encoded_cols].copy()
for col in numeric_features.columns:
    numeric_features[col] = pd.to_numeric(numeric_features[col], errors='coerce')

numeric_features = numeric_features.replace([np.inf, -np.inf], np.nan)
numeric_features = numeric_features.fillna(numeric_features.median(numeric_only=True)).fillna(0)

# ── Drop near-zero-variance columns (< 0.5% non-zero) ──
nonzero_frac = (numeric_features != 0).mean()
low_variance_cols = nonzero_frac[nonzero_frac < 0.005].index.tolist()
numeric_features = numeric_features.drop(columns=low_variance_cols)

features_encoded = numeric_features.copy()
features_encoded.columns = sanitize_columns(features_encoded.columns)
features_encoded = features_encoded.loc[:, ~features_encoded.columns.duplicated()].copy()

print('Target-encoded categorical features:', len(target_encoded_cols))
print('Dropped near-zero-variance columns:', len(low_variance_cols))
print('Final encoded feature matrix:', features_encoded.shape)
display(features_encoded.head())

Target-encoded categorical features: 23
Dropped near-zero-variance columns: 2
Final encoded feature matrix: (8173, 89)


,latitude,longitude,valid_start_coordinate,has_start_location,distance_to_city_center_km,start_hour,start_dayofweek,start_weekofyear,is_weekend,is_peak_hour,is_night,hour_sin,hour_cos,day_sin,day_cos,report_lag_minutes_clipped,report_lag_hours_clipped,report_lag_is_negative,reporting_delay_minutes,text_length,description_char_length,description_word_count,has_non_ascii_text,has_kannada_text,has_accident_word,has_breakdown_word,has_water_word,has_construction_word,has_event_word,has_blocked_word,has_jam_word,has_location_hint_word,has_emergency_word,is_planned_event,is_public_or_vip_event,is_breakdown_event,is_accident_event,is_weather_or_visibility_event,is_road_condition_event,has_vehicle_type,is_truck,is_bus,is_heavy_vehicle,has_cargo_material,age_of_truck,truck_age_missing,past_count_event_cause,past_closure_rate_event_cause,past_count_corridor,past_closure_rate_corridor,past_count_zone,past_closure_rate_zone,past_count_junction,past_closure_rate_junction,past_count_police_station,past_closure_rate_police_station,past_count_location_grid,past_closure_rate_location_grid,past_count_event_cause_corridor,past_closure_rate_event_cause_corridor,cause_base_closure_rate,cause_severity_rank,corridor_is_major,priority_is_high,description_urgency_score,is_authenticated,location_grid_te,event_type_grouped_te,event_cause_grouped_te,direction_grouped_te,veh_type_grouped_te,corridor_grouped_te,priority_grouped_te,cargo_material_grouped_te,reason_breakdown_grouped_te,police_station_grouped_te,zone_grouped_te,junction_grouped_te,authenticated_grouped_te,truck_age_band_te,peak_period_te,event_cause_corridor_te,cause_peak_interaction_te,zone_cause_interaction_te,corridor_cause_interaction_te,cause_heavy_vehicle_interaction_te,corridor_peak_interaction_te,start_day_name_te,start_month_te
0,13.040004,77.518099,1,1,11.249443,22.0,3.0,10.0,0,0,1,-0.500000,0.866025,0.433884,-0.900969,2.050884,0.034181,0,2.050884,132,31,7,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,1,0,0,1,0,10.0,1,3750.0,0.046400,343.0,0.029155,2591.0,0.083365,3525.0,0.076879,139.0,0.035971,14.0,0.071429,288.0,0.017361,0.04,8.0,1,1,1,1,0.075681,0.065858,0.043135,0.082289,0.028928,0.029675,0.059186,0.083699,0.083699,0.045583,0.087514,0.082818,0.083724,0.083196,0.066252,0.020536,0.045649,0.038961,0.020536,0.043719,0.028058,0.082472,0.090598
1,12.921876,77.645158,1,1,7.783945,9.0,1.0,5.0,0,1,0,0.707107,-0.707107,0.781831,0.623490,0.979700,0.016328,0,0.979700,108,16,2,0,0,0,1,0,0,0,0,0,1,0,0,0,1,0,0,0,1,0,0,1,0,10.0,1,2594.0,0.048188,113.0,0.106195,690.0,0.068116,1628.0,0.062654,56.0,0.125000,3748.0,0.077375,71.0,0.056338,0.04,8.0,1,1,2,1,0.085206,0.065858,0.043135,0.082289,0.043700,0.074749,0.059186,0.083699,0.083699,0.085489,0.087514,0.082818,0.083724,0.083196,0.061801,0.046094,0.041049,0.038961,0.046094,0.043719,0.066886,0.072662,0.075348
2,12.955622,77.585708,1,1,2.021119,11.0,5.0,45.0,1,0,0,0.258819,-0.965926,-0.974928,-0.222521,1.960773,0.032680,0,1.960773,219,117,15,1,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,10.0,1,8.0,0.000000,28.0,0.107143,15.0,0.066667,47.0,0.063830,3.0,0.000000,62.0,0.064516,3.0,0.000000,0.09,6.0,0,0,1,1,0.085206,0.065858,0.086050,0.082289,0.141882,0.120635,0.120303,0.083699,0.083699,0.151714,0.086495,0.086241,0.083724,0.083196,0.081054,0.095421,0.076709,0.063004,0.095421,0.086050,0.118321,0.068888,0.056108
3,13.006147,77.579435,1,1,4.178109,23.0,3.0,10.0,0,0,1,-0.258819,0.965926,0.433884,-0.900969,2.027265,0.033788,0,2.027265,96,9,2,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,10.0,1,101.0,0.425743,2286.0,0.111549,2592.0,0.083333,3526.0,0.076858,209.0,0.066986,5462.0,0.081289,70.0,0.500000,0.39,3.0,0,0,1,1,0.085206,0.065858,0.364590,0.082289,0.141882,0.120635,0.120303,0.083699,0.083699,0.073739,0.087514,0.082818,0.083724,0.083196,0.066252,0.409418,0.298163,0.356592,0.409418,0.364590,0.098214,0.082472,0.090598
4,12.953980,77.585233,1,1,2.206553,10.0,1.0,5.0,0,0,0,0.500000,-0.866025,0.781831,0.623490,2.393161,0.039886,0,2.393161,151,48,7,1,1,0,0,0,0,0,0,0,1,0,0,0,1,0,

## 9. Build Model-Ready Tables

Model 1 uses all labeled road-closure rows.

Model 2 uses rows with valid duration labels.

In [9]:
road_closure_df = features_encoded.copy()
road_closure_df['target_road_closure'] = pd.to_numeric(df['target_road_closure'], errors='coerce').astype(int)

# Keep identifiers only in separate audit files, not model-ready matrices.
duration_df = features_encoded.loc[df['valid_duration_label'].eq(1)].copy()
duration_df['duration_band'] = df.loc[df['valid_duration_label'].eq(1), 'duration_band'].values

print('Road closure model-ready shape:', road_closure_df.shape)
print('Duration-band model-ready shape:', duration_df.shape)
print('\nRoad closure target:')
display(road_closure_df['target_road_closure'].value_counts().to_frame('count'))
print('\nDuration band target:')
display(duration_df['duration_band'].value_counts().to_frame('count'))

Road closure model-ready shape: (8173, 90)
Duration-band model-ready shape: (2764, 90)

Road closure target:


,count
target_road_closure,
0,7497
1,676



Duration band target:


,count
duration_band,
short,1581
medium,903
long,280


## 10. Leakage Sanity Check

This checks that known future/leakage fields did not survive into the encoded feature matrices.

In [10]:
forbidden_fragments = [
    'closed_datetime', 'resolved_datetime', 'end_datetime', 'modified_datetime',
    'endlatitude', 'endlongitude', 'resolved_at_latitude', 'resolved_at_longitude',
    'closed_by', 'resolved_by', 'status', 'duration_minutes', 'route_path',
    'end_address', 'has_end_location', 'route_distance', 'has_route_span',
]

feature_names = [c for c in road_closure_df.columns if c != 'target_road_closure']
leakage_hits = [c for c in feature_names if any(fragment in c for fragment in forbidden_fragments)]

print('Leakage hits:', leakage_hits)
assert not leakage_hits, f'Potential leakage columns found: {leakage_hits}'

Leakage hits: []


## 11. Save Outputs

In [11]:
road_closure_path = OUTPUT_DIR / 'model_ready_road_closure.csv'
duration_path = OUTPUT_DIR / 'model_ready_duration_band.csv'
feature_columns_path = OUTPUT_DIR / 'feature_columns.json'
summary_path = OUTPUT_DIR / 'feature_engineering_summary.csv'

road_closure_df.to_csv(road_closure_path, index=False)
duration_df.to_csv(duration_path, index=False)

feature_metadata = {
    'feature_count_road_closure': int(road_closure_df.shape[1] - 1),
    'feature_count_duration_band': int(duration_df.shape[1] - 1),
    'road_closure_feature_columns': [c for c in road_closure_df.columns if c != 'target_road_closure'],
    'duration_feature_columns': [c for c in duration_df.columns if c != 'duration_band'],
    'categorical_source_columns': categorical_feature_cols,
    'numeric_source_columns': numeric_feature_cols,
}
with open(feature_columns_path, 'w') as f:
    json.dump(feature_metadata, f, indent=2)

summary = pd.DataFrame({
    'metric': [
        'road_closure_rows', 'road_closure_columns', 'road_closure_features',
        'duration_rows', 'duration_columns', 'duration_features',
        'duration_short_rows', 'duration_medium_rows', 'duration_long_rows',
    ],
    'value': [
        len(road_closure_df), road_closure_df.shape[1], road_closure_df.shape[1] - 1,
        len(duration_df), duration_df.shape[1], duration_df.shape[1] - 1,
        int(duration_df['duration_band'].eq('short').sum()),
        int(duration_df['duration_band'].eq('medium').sum()),
        int(duration_df['duration_band'].eq('long').sum()),
    ]
})
summary.to_csv(summary_path, index=False)

print('Saved:', road_closure_path)
print('Saved:', duration_path)
print('Saved:', feature_columns_path)
print('Saved:', summary_path)
display(summary)

Saved: D:\Python\Gridlock\Phase 2\theme 2\outputs\feature_engineering\model_ready_road_closure.csv
Saved: D:\Python\Gridlock\Phase 2\theme 2\outputs\feature_engineering\model_ready_duration_band.csv
Saved: D:\Python\Gridlock\Phase 2\theme 2\outputs\feature_engineering\feature_columns.json
Saved: D:\Python\Gridlock\Phase 2\theme 2\outputs\feature_engineering\feature_engineering_summary.csv


,metric,value
0,road_closure_rows,8173
1,road_closure_columns,90
2,road_closure_features,89
3,duration_rows,2764
4,duration_columns,90
5,duration_features,89
6,duration_short_rows,1581
7,duration_medium_rows,903
8,duration_long_rows,280


## 12. Block 2 Completion Checklist

Before moving to Block 3:

- `model_ready_road_closure.csv` exists.
- `model_ready_duration_band.csv` exists.
- Leakage sanity check passes.
- Target distributions match Block 1.

In [12]:
assert road_closure_path.exists(), road_closure_path
assert duration_path.exists(), duration_path
assert feature_columns_path.exists(), feature_columns_path
assert summary_path.exists(), summary_path
assert road_closure_df['target_road_closure'].notna().all()
assert duration_df['duration_band'].notna().all()

print('Block 2 complete.')
print('Next block: Model 1 road-closure classifier.')

Block 2 complete.
Next block: Model 1 road-closure classifier.
